# 09 · Learning the scale factor f (1P1Q)

In the 1P1Q paper, $f$ is trained together with the VQC parameters by gradient descent.
QSVC has no gradient-trained circuit, so the kernel-method equivalent is **selecting $f$ by cross-validation on the training subsample**:
for every candidate $f$, 5-fold CV AUC on the 400 training jets; the $f$ with the best CV AUC is used to train on all 400 jets and is evaluated **once** on the test set.

The test set is never used to choose $f$ (notebooks 07–08 picked the best $f$ by looking at test AUC, which is slightly optimistic).

Encodings: original 1P1Q ($\Delta\eta, \Delta\phi$; $|\psi_i\rangle = R_X(\phi_i) R_Y(\theta_i)|0\rangle$) and the rotation-invariant variant ($z\,\Delta R$; $R_Y$), both with $N = 10$ constituents.

> **Note added after the independent review (11 Nov 2026).**

C is fixed at the scikit-learn default (C = 1) in this notebook. At small f the invariant kernel is ≈ 1 − f²‖Δa‖²/4, so f and C are nearly degenerate and the "learned f" is poorly identified (notebook 10, with C tuned, chose different values). Quote f only together with C.

In [1]:
import sys
sys.path.insert(0, "src")

import json
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

from jet_data import (load_jets, select_quark_gluon, event_level_split, balanced_subsample,
                      leading_constituents, BOOKKEEPING, PF_KEYS)
from kernels import onep1q_states, onep1q_kernel, angle_kernel, shot_noise

SEED = 42
SEEDS = [0, 1, 2, 3, 4]
N_PER_CLASS = 200
SHOTS = 1000
N_CONST = 10
F_CANDIDATES = np.round(np.logspace(0, 2.5, 12), 2)        # 1 ... 316
RESULTS = Path("results")

raw = load_jets(features=("jetPt",), extra=BOOKKEEPING + PF_KEYS)
jets = select_quark_gluon(raw, require_loose_id=True)
y = jets["label"]
train, test = event_level_split(jets, test_fraction=0.3, seed=SEED)
z, d_eta, d_phi, _ = leading_constituents(jets, n_max=N_CONST)
d_r = np.sqrt(d_eta**2 + d_phi**2)
test_idx = np.flatnonzero(test)

def kernel_fn(name, f):
    if name == "1P1Q (original)":
        st = lambda r: onep1q_states(z[r], d_eta[r], d_phi[r], f)
        return lambda ra, rb: onep1q_kernel(st(ra), st(rb))
    ang = lambda r: f * z[r] * d_r[r]
    return lambda ra, rb: angle_kernel(ang(ra), ang(rb))

ENCODINGS = ["1P1Q (original)", "1P1Q rotation-invariant (ours)"]
print("candidate f:", F_CANDIDATES)

candidate f: [  1.     1.69   2.85   4.81   8.11  13.69  23.1   38.99  65.79 111.03
 187.38 316.23]


In [2]:
def cv_auc(K_full, labels, folds):
    aucs = []
    for tr, va in folds:
        svc = SVC(kernel="precomputed").fit(K_full[np.ix_(tr, tr)], labels[tr])
        aucs.append(roc_auc_score(labels[va], svc.decision_function(K_full[np.ix_(va, tr)])))
    return np.mean(aucs)

results = {e: [] for e in ENCODINGS}
for enc in ENCODINGS:
    for s in SEEDS:
        idx = balanced_subsample(y, train, N_PER_CLASS, seed=s)
        folds = list(StratifiedKFold(5, shuffle=True, random_state=s).split(idx, y[idx]))
        cv = []
        for f in F_CANDIDATES:
            K = kernel_fn(enc, f)(idx, idx)
            cv.append(cv_auc(K, y[idx], folds))
        f_best = float(F_CANDIDATES[int(np.argmax(cv))])
        kf = kernel_fn(enc, f_best)
        K_tr, K_te = kf(idx, idx), kf(test_idx, idx)
        auc_exact = roc_auc_score(y[test_idx], SVC(kernel="precomputed").fit(K_tr, y[idx]).decision_function(K_te))
        rng = np.random.default_rng(1000 + s)
        svc = SVC(kernel="precomputed").fit(shot_noise(K_tr, SHOTS, rng, symmetric=True), y[idx])
        auc_shots = roc_auc_score(y[test_idx], svc.decision_function(shot_noise(K_te, SHOTS, rng)))
        results[enc].append({"seed": s, "f": f_best, "cv_auc": float(max(cv)), "cv_curve": [float(c) for c in cv],
                             "auc_exact": auc_exact, "auc_shots": auc_shots})
        print(f"{enc:32s} seed {s}: learned f = {f_best:7.2f}, CV AUC {max(cv):.3f}, test AUC {auc_exact:.3f} (exact) / {auc_shots:.3f} ({SHOTS} shots)")

json.dump({"f_candidates": F_CANDIDATES.tolist(), "results": results}, open(RESULTS / "learned_f.json", "w"), indent=2)
print("saved results/learned_f.json")

1P1Q (original)                  seed 0: learned f =  316.23, CV AUC 0.656, test AUC 0.687 (exact) / 0.685 (1000 shots)
1P1Q (original)                  seed 1: learned f =  111.03, CV AUC 0.673, test AUC 0.688 (exact) / 0.679 (1000 shots)
1P1Q (original)                  seed 2: learned f =  187.38, CV AUC 0.645, test AUC 0.686 (exact) / 0.681 (1000 shots)
1P1Q (original)                  seed 3: learned f =  187.38, CV AUC 0.698, test AUC 0.683 (exact) / 0.681 (1000 shots)
1P1Q (original)                  seed 4: learned f =  187.38, CV AUC 0.702, test AUC 0.683 (exact) / 0.682 (1000 shots)
1P1Q rotation-invariant (ours)   seed 0: learned f =   23.10, CV AUC 0.670, test AUC 0.701 (exact) / 0.698 (1000 shots)
1P1Q rotation-invariant (ours)   seed 1: learned f =   65.79, CV AUC 0.714, test AUC 0.715 (exact) / 0.711 (1000 shots)
1P1Q rotation-invariant (ours)   seed 2: learned f =   65.79, CV AUC 0.705, test AUC 0.712 (exact) / 0.705 (1000 shots)
1P1Q rotation-invariant (ours)   seed 3:

## Summary

In [3]:
for enc in ENCODINGS:
    r = results[enc]
    fs = [x["f"] for x in r]
    ae = [x["auc_exact"] for x in r]; ash = [x["auc_shots"] for x in r]
    print(f"- {enc}: learned f median {np.median(fs):.1f} (range {min(fs):.1f}-{max(fs):.1f}); "
          f"test AUC {np.mean(ae):.3f} ± {np.std(ae):.3f} (exact), {np.mean(ash):.3f} ± {np.std(ash):.3f} ({SHOTS} shots)")

- 1P1Q (original): learned f median 187.4 (range 111.0-316.2); test AUC 0.686 ± 0.002 (exact), 0.682 ± 0.002 (1000 shots)
- 1P1Q rotation-invariant (ours): learned f median 39.0 (range 23.1-65.8); test AUC 0.711 ± 0.005 (exact), 0.705 ± 0.006 (1000 shots)
